# The 5D Parallelism Landscape

> The previous section walked through collective communication primitives: all-reduce sums gradients, all-gather stitches sharded parameters back together, and reduce-scatter aggregates gradients after slicing them. With this vocabulary in hand, distributed training schemes no longer look like a tangle of framework abstractions — at heart, every parallelism strategy is just "which tensor axes to split, and which collectives to run at which moments".
>
> This section lays out all five parallelism dimensions used in industrial training: Data Parallelism, Tensor Parallelism, Pipeline Parallelism, Sequence Parallelism, and Expert Parallelism. For each one we make it clear "which axis is split, when communication happens, and why splitting this way saves memory or bandwidth", and finally look at how they compose into the real training configs of Llama-3, DeepSeek-V3, and Mixtral.


Data Parallelism (DP) only splits the batch: every card holds a full replica of the model, and gradients are all-reduced after the backward pass. It scales throughput linearly to N times, but saves no memory. ZeRO shards the redundancy inside DP (optimizer states, gradients, parameters) across cards layer by layer, dropping per-card memory to 16P/N at the cost of extra all-gather and reduce-scatter operations. This part was covered earlier; this appendix only recaps the conclusion as a refresher.

The real key to scaling model size is model parallelism — splitting a single layer's weights along some dimension. Tensor Parallelism (TP) splits the hidden dimension, communicating once with an all-reduce per layer in the forward pass and once more in the backward; Megatron-LM pushed this to industrial usability. Pipeline Parallelism (PP) splits layers, with each card owning a contiguous run of transformer blocks and using micro-batch pipelining to hide communication; DeepSeek-V3's DualPipe turns the pipeline bidirectional to squeeze the bubble further. Sequence Parallelism (SP) splits the sequence length, in support of long-context training. Expert Parallelism (EP) is specific to MoE, assigning different experts to different cards and exchanging tokens with all-to-all during routing.

This appendix carries over the partition notation from the previous section: $I_X$ denotes the $I$ axis of a tensor sharded along the $X$ axis of the device mesh. The code simulates multiple cards in a single process using numpy, with each numpy array standing in for the data on one card. The emphasis is on letting the reader see "how the shape gets split".


## 1. Data Parallelism and ZeRO: A Quick Recap

DDP has four steps: each card runs the forward and backward on its own micro-batch to compute gradients, all-reduce averages the gradients, and each card independently updates the full parameters. Communication is just a single all-reduce at the end of the backward. When the model fits on a single card, DDP is the optimal choice.

When the model does not fit, ZeRO kicks in. The three stages shard optimizer states, gradients, and parameters respectively. Under Stage 3, each card holds only $1/N$ of the parameters, gradients, and optimizer states, all-gathering the full weights of the current layer on demand during the forward and discarding them once used. A key fact is that **all three stages have identical communication volume**. The reason is that all-reduce is equivalent to reduce-scatter plus all-gather: ZeRO-1 reduce-scatters the gradients across cards, and Stage 3 all-gathers the parameters back, with each of the two operations accounting for half the traffic, summing to exactly one all-reduce. So from a communication-cost standpoint, Stages 1/2/3 are free — they simply trade the reclaimed bandwidth for memory savings.

The cell below revisits the per-card memory of the three stages for a 7B model on 8 cards, as the starting point of this section.


In [ ]:
# === ZeRO three stages: per-card memory recap, 7B x 8 cards ===
P = 7e9
N = 8

ddp_per_card   = 16 * P                       # full replica per card
zero1_per_card = 2 * P + 2 * P + 12 * P / N   # shard optimizer only
zero2_per_card = 2 * P + 2 * P / N + 12 * P / N
zero3_per_card = 16 * P / N                   # shard everything

print(f"{'Scheme':<14}{'Per-card (GB)':>14}{'vs DDP':>12}")
print("-" * 40)
for name, val in [("DDP",        ddp_per_card),
                  ("ZeRO-1",     zero1_per_card),
                  ("ZeRO-2",     zero2_per_card),
                  ("ZeRO-3",     zero3_per_card)]:
    print(f"{name:<14}{val/1e9:>12.1f}   {val/ddp_per_card*100:>8.1f}%")
print()
print("Key observation: all three stages have identical communication volume, yet per-card memory drops from 112 GB to 14 GB.")
print("The tradeoff is that higher stages are more bandwidth-sensitive and demand a higher compute-to-communication ratio.")


## 2. Tensor Parallelism: Splitting the Hidden Dimension

The core idea of Tensor Parallelism (TP) is to split the weight matrix of a single linear operation along some dimension, so each card only computes its own slice. There are two ways to split: ColumnParallelLinear splits the weight matrix by columns (i.e. by the output dimension), and each card computes a partial set of columns for the full batch with no communication needed; RowParallelLinear splits the weight by rows (i.e. by the input dimension), and each card computes a partial sum that needs an all-reduce to produce the complete result.

The key insight of Megatron-LM is to chain these two splits together. An MLP consists of two linear layers, $W_1$ and $W_2$: $W_1$ uses column parallel and $W_2$ uses row parallel, with the nonlinear activation in between applied independently on each card's local columns. That way the entire MLP layer requires only a single all-reduce at the very end, from input to output. The hand calculation below shows why this property holds.

Let $h$ be the input hidden state (replicated to every card), $W_1$ column-split into $[W_{1,0}; W_{1,1}]$, and $W_2$ row-split into $[W_{2,0}, W_{2,1}]^\top$. After the first layer each card has $h_i = h \cdot W_{1,i}$ (local, no communication); the activation acts elementwise, $\sigma(h_i)$ (still local); the second layer gives $y = \sigma(h) \cdot W_2 = [\sigma(h_0), \sigma(h_1)] \cdot [W_{2,0}; W_{2,1}] = \sigma(h_0) \cdot W_{2,0} + \sigma(h_1) \cdot W_{2,1}$. The two slices sum to exactly the full product, so a single all-reduce suffices.


In [ ]:
# === TP hand calculation: 2 cards, verify column -> row needs only one all-reduce ===
import numpy as np
np.random.seed(0)

d_model = 4     # input dimension (small numbers for hand calculation)
d_ff    = 6     # intermediate dimension
world_size = 2  # 2 cards

# Build the full weights (reference implementation)
W1_full = np.random.randn(d_model, d_ff)      # (4, 6)
W2_full = np.random.randn(d_ff, d_model)      # (6, 4)
h       = np.random.randn(1, d_model)         # (1, 4)

def silu(x):
    return x / (1.0 + np.exp(-x))

# Reference output: full computation on a single card (with silu activation, matching the TP path)
y_ref = silu(h @ W1_full) @ W2_full           # (1, 4)

# TP sharding: W1 split by columns (output dim), W2 split by rows (input dim)
W1_shards = np.split(W1_full, world_size, axis=1)   # 2 shards, each (4, 3)
W2_shards = np.split(W2_full, world_size, axis=0)   # 2 shards, each (3, 4)

# Simulate each card's local computation
partial = []
for rank in range(world_size):
    h_local = silu(h @ W1_shards[rank])       # (1, 3) - no communication, activation applied locally
    y_partial = h_local @ W2_shards[rank]     # (1, 4) - partial sum
    partial.append(y_partial)
    print(f"rank {rank}: y_partial shape = {y_partial.shape}, values = {y_partial.ravel()}")

# Simulate a single all-reduce (sum)
y_tp = sum(partial)
print()
print(f"Reference output y_ref = {y_ref.ravel()}")
print(f"TP reconstruction y_tp = {y_tp.ravel()}")
print(f"Max error = {np.abs(y_ref - y_tp).max():.2e}")
print()
print("Key observation: the entire column -> activation -> row path performs only one all-reduce (the sum).")


### 2.1 Splitting Attention by Head

In Transformer multi-head attention, each head is independent — head A's Q only dots with head A's K and V. This means that if we shard QKV across cards by head, each card can independently complete the attention computation for its own batch of heads, with no communication needed in between. This is the most elegant aspect of applying TP to attention: column parallel (shard QKV by head) -> local attention -> row parallel (output projection), again with only a single all-reduce at the end.

In practice the number of heads must be divisible by world_size. For example, with 32 heads and 4-way TP, each card handles 8 heads, and each card's QKV projection weights drop from $d_\text{model} \times 3d_\text{model}$ to $d_\text{model} \times 3 d_\text{model}/4$.


In [ ]:
# === Attention TP: split by head, verify communication-free ===
# Reference uses the standard formulation: compute each head independently, concatenate, then project
d_model, num_heads = 8, 4
head_dim = d_model // num_heads      # 2
seq_len  = 3
world_size = 2                       # 2 cards, 2 heads per card

np.random.seed(2)
x = np.random.randn(1, seq_len, d_model)

# Per-head q/k/v/out weights (equivalent to standard MHA written this way)
W_qkv_per_head = np.random.randn(num_heads, d_model, 3 * head_dim)  # (4, 8, 6)
W_out_per_head = np.random.randn(num_heads, head_dim, d_model)      # (4, 2, 8)

def softmax(x, axis=-1):
    e = np.exp(x - x.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)

def head_forward(x, h_id):
    """Full forward pass for a single head."""
    qkv = x @ W_qkv_per_head[h_id]               # (1, 3, 6) -> reshape
    qkv = qkv.reshape(1, seq_len, 3, head_dim)
    q, k, v = qkv[..., 0, :], qkv[..., 1, :], qkv[..., 2, :]
    # (1, seq, head_dim) -> (1, head_dim, seq) via transpose for q
    q = q.transpose(0, 2, 1)                     # (1, 2, 3)
    k = k.transpose(0, 2, 1)                     # (1, 2, 3)
    v = v.transpose(0, 2, 1)                     # (1, 2, 3)
    sc = q @ k.transpose(0, 2, 1) / np.sqrt(head_dim)
    at = softmax(sc) @ v                         # (1, 2, 3)
    at = at.transpose(0, 2, 1)                   # (1, 3, 2)
    return at @ W_out_per_head[h_id]             # (1, 3, 8) partial sum

# Reference: sum over all heads (equivalent to full MHA)
y_ref = sum(head_forward(x, h) for h in range(num_heads))

# TP sharding: assign heads to 2 cards, 2 heads each, zero communication
heads_per_rank = num_heads // world_size
partials = []
for rank in range(world_size):
    local_heads = range(rank * heads_per_rank, (rank + 1) * heads_per_rank)
    partial = sum(head_forward(x, h) for h in local_heads)
    partials.append(partial)

y_tp = sum(partials)
print(f"Reference output y_ref[0,0,:] = {y_ref[0,0,:]}")
print(f"TP reconstruction y_tp [0,0,:] = {y_tp[0,0,:]}")
print(f"Max error = {np.abs(y_ref - y_tp).max():.2e}")
print()
print("Key observation: each card independently computes the heads it owns; attention is communication-free inside.")
print("A single all-reduce at the end sums the per-card partial results.")


### 2.2 Three-Matrix Sharding for the SwiGLU MLP

The SwiGLU MLP used by the Llama family has three weight matrices: $W_\text{gate}$, $W_\text{up}$, and $W_\text{down}$. The forward pass is $\text{SiLU}(x W_\text{gate}) \odot (x W_\text{up})$ multiplied by $W_\text{down}$. Both $W_\text{gate}$ and $W_\text{up}$ are column parallel (split by columns, each card computing its own batch of intermediate-dimension columns); their outputs are multiplied elementwise and remain sharded along the intermediate dimension. $W_\text{down}$ uses row parallel, with a single all-reduce at the end. This is exactly the same sharding as an ordinary MLP — there is just an extra elementwise product in the middle.

In a MoE model, the MLP inside each expert uses the same sharding. That is, TP applies to both the MLP of a dense model and the MLP of a single expert — a fact we will reuse in Section 5 on Expert Parallelism.


In [ ]:
# === SwiGLU MLP TP sharding: gate + up column, down row ===
d_model, d_ff = 4, 6
world_size = 2

W_gate = np.random.randn(d_model, d_ff)
W_up   = np.random.randn(d_model, d_ff)
W_down = np.random.randn(d_ff, d_model)
x      = np.random.randn(1, d_model)

def silu(z):
    return z / (1.0 + np.exp(-z))

# Reference: full computation on a single card
y_ref = (silu(x @ W_gate) * (x @ W_up)) @ W_down

# TP sharding
gate_sh = np.split(W_gate, world_size, axis=1)   # column
up_sh   = np.split(W_up,   world_size, axis=1)   # column
down_sh = np.split(W_down, world_size, axis=0)   # row

partials = []
for rank in range(world_size):
    g = x @ gate_sh[rank]                # (1, 3) local
    u = x @ up_sh[rank]                  # (1, 3) local
    h = silu(g) * u                       # elementwise, still local
    y_partial = h @ down_sh[rank]        # (1, 4) partial sum
    partials.append(y_partial)

y_tp = sum(partials)
print(f"y_ref = {y_ref.ravel()}")
print(f"y_tp  = {y_tp.ravel()}")
print(f"Max error = {np.abs(y_ref - y_tp).max():.2e}")
print("SwiGLU three matrices: gate/up column + down row, a single all-reduce at the end.")


### 2.3 The Communication Cost of TP

The upside of TP is that per-card memory drops linearly with world_size — the weights are split into N shards, and each card holds only $1/N$. The downside is that communication is frequent and always on the critical path: the MLP and Attention of every transformer layer each perform one forward all-reduce and one backward all-reduce, for a total of four. These all-reduces are latency-sensitive across nodes, so TP is typically confined to within a single node (8 cards connected by NVLink), with PP or DP handling the cross-node dimension.

The table below compares per-layer MLP weights and communication counts for TP=1, TP=2, and TP=4 (d_model=4096, d_ff=11008):


In [ ]:
# === TP scale vs per-card weights / communication count ===
d_model, d_ff = 4096, 11008
bytes_per_elem = 2   # FP16/BF16

print(f"{'TP':>4}{'Per-card MLP weights (MB)':>28}{'all-reduce count/layer':>26}")
print("-" * 60)
for tp in [1, 2, 4, 8]:
    w_bytes = (d_model * d_ff + d_ff * d_model) * bytes_per_elem / tp
    comms = 4 if tp > 1 else 0   # fwd + bwd x (MLP + Attn), no collectives when TP=1
    print(f"{tp:>4}{w_bytes / 1e6:>26.1f}{comms:>26}")
print()
print("Observation: at TP=8 per-card MLP weights shrink to 1/8, but each layer still does 4 all-reduces.")
print("Bandwidth within an NVLink domain is sufficient; cross-node TP is usually not worth it — use PP or DP instead.")


## 3. Pipeline Parallelism: Splitting Layers

The most naive form of model parallelism places each layer on a different card: during the forward pass the hidden state is shipped from card 0 to card 1 then to card 2, and so on. The problem with this naive approach is that at any moment only one card is computing and the rest are idle. Splitting 4 layers across 4 cards yields the same throughput as a single card — only the memory is distributed — which is clearly pointless.

GPipe's idea is to split a mini-batch into M micro-batches and let them execute in staggered fashion through the pipeline: once card 0 finishes micro-batch 0 it immediately starts micro-batch 1, while card 1 works on micro-batch 0. That way most of the time every card is busy, with bubbles only at the start and end of the pipeline. The bubble fraction is $(P-1)/M$, where $P$ is the number of stages (cards) and $M$ is the number of micro-batches. Larger $M$ means a smaller bubble.

The 1F1B (one forward, one backward) schedule inserts the backward pass into the pipeline on top of GPipe: each stage, immediately after finishing the forward of one micro-batch, runs the backward of an earlier micro-batch, which lowers the peak activation memory while still keeping the pipeline full.


In [ ]:
# === GPipe bubble fraction hand calculation ===
print(f"{'P (stages)':>12}{'M (micro)':>12}{'bubble fraction':>18}")
print("-" * 44)
for P in [4, 8, 16]:
    for M in [8, 32, 128]:
        bubble = (P - 1) / M
        print(f"{P:>12}{M:>12}{bubble*100:>16.1f}%")
print()
print("Observation: at P=8, M=128 the bubble is only 5.5%, essentially negligible.")
print("In practice M is usually chosen to be 4-8x the number of stages to keep the bubble low.")


### 3.1 The 1F1B Pipeline Timeline

The key design of 1F1B is to start the backward pass on each stage as early as possible. The ASCII diagram below shows the 1F1B timeline for P=4, M=4: each letter denotes a forward (a number) or a backward (a number with a prime) of a micro-batch. Stage 0 is on the far left (earliest), stage 3 on the far right.


In [ ]:
# === 1F1B timeline ASCII art (P=4 stages, M=4 micro-batches) ===
P, M = 4, 4
print("1F1B schedule (F = forward, B = backward, number is the micro-batch index)")
print("=" * 60)
# Simplified steady-state segment: once a stage starts, it interleaves F and B
for stage in range(P):
    # warmup: the further back the stage, the more micro-batches it must wait for upstream
    warmup = stage
    slots = [" . "] * (warmup + M + (P - 1 - stage))
    # forward segment
    for mb in range(M):
        idx = warmup + mb
        if idx < len(slots):
            slots[idx] = f"F{mb}"
    # backward segment: immediately after the last forward
    for mb in range(M):
        idx = warmup + M + mb
        if idx < len(slots):
            slots[idx] = f"B{mb}"
    print(f"stage {stage}: " + " ".join(slots))
print()
print("Key observation: during the steady-state segment every card is busy; there is a warmup bubble at the start and a cooldown bubble at the end.")
print("These are exactly the two end bubbles captured by the (P-1)/M formula.")


### 3.2 DualPipe: Bidirectional Pipelining (DeepSeek-V3)

1F1B still has bubbles at both ends. DualPipe, proposed in DeepSeek-V3, splits the micro-batches in half and injects them from both ends of the pipeline: one half flows from stage 0 to stage P-1, the other in reverse. The two pipelines interleave on the same set of cards, with each card simultaneously running a forward in one direction and a backward in the other. The bubble can then be hidden by the computation of micro-batches flowing the opposite way, further lowering the bubble fraction.

The cost is that each card must hold activations for both directions at once, so the peak memory is higher than with unidirectional 1F1B. When DeepSeek-V3 trained on 2048 H800s with PP=16, DualPipe pushed the bubble fraction down to the 1/P range, and together with EP+DP completed a pretraining run of 14.8T total tokens. This is currently the representative PP scheduling scheme in industry.


In [ ]:
# === DualPipe vs 1F1B bubble comparison (simplified estimate) ===
print(f"{'P':>4}{'M':>6}{'1F1B bubble':>16}{'DualPipe bubble':>20}")
print("-" * 48)
for P in [4, 8, 16]:
    M = 4 * P   # in practice M is often 4P
    one_f1b = (P - 1) / M
    dual    = (P - 1) / (2 * M)   # bidirectional doubles the effective number of micro-batches
    print(f"{P:>4}{M:>6}{one_f1b*100:>14.1f}%{dual*100:>18.1f}%")
print()
print("DualPipe doubles the effective number of micro-batches, halving the bubble. The cost is doubled activation memory.")


## 4. Sequence Parallelism: Splitting the Sequence

Sequence Parallelism (SP) splits the sequence-length dimension. It has two uses. The first is in support of TP: in Megatron-LM's TP, operations like LayerNorm and dropout act on each token independently, so there is no need for every card to hold the full sequence — shard the sequence dimension as well, let each card run LayerNorm only on its own batch of tokens, all-gather the sequence back together before entering TP's column/row parallel regions, and reduce-scatter to shard it again on exit. This shards the activation memory of LayerNorm and dropout by a factor of N, which significantly saves memory during long-sequence training.

The second use is genuine long-context training: when seq_len exceeds the per-card memory limit (say 128K or 1M context), even if the model itself fits, the intermediate activations of attention blow up the memory. In this case the sequence is sharded across cards, and during the attention computation techniques like ring attention or all-to-all let each card see how its segment of the sequence relates to the other segments. Methods in this family include Ring Attention and Context Parallelism (DeepSpeed-Ulysses); they are essentially finer-grained applications of SP inside attention.


In [ ]:
# === SP: effect of sequence sharding on LayerNorm activation memory ===
batch, seq_len, d_model = 1, 8192, 4096
bytes_per_elem = 2

full_act_bytes = batch * seq_len * d_model * bytes_per_elem

print(f"Full LayerNorm input activation: {full_act_bytes / 1e6:.1f} MB")
print(f"{'SP':>6}{'Per-card activation (MB)':>26}{'Savings':>12}")
print("-" * 46)
for sp in [1, 2, 4, 8]:
    per_card = full_act_bytes / sp
    print(f"{sp:>6}{per_card / 1e6:>24.1f}{(1 - 1/sp)*100:>10.0f}%")
print()
print("Observation: at SP=8 the per-card LayerNorm activation shrinks to 1/8.")
print("Cost: an all-gather to reassemble before entering TP, and a reduce-scatter to shard again on exit.")


## 5. Expert Parallelism: Specific to MoE

Expert Parallelism (EP) is specific to Mixture-of-Experts models. The core structure of MoE is a set of parallel experts (each an MLP), with a router selecting the top-k experts for each token. EP's approach is to distribute N experts across N cards, with each card holding the weights of just one (or a few) experts. This is the exact opposite of DP, which replicates the same expert to all cards.

The core communication in EP is the all-to-all after the router: the experts chosen for each token are spread across different cards, so during the forward pass tokens must be sent to the cards hosting their assigned experts, and after computation an all-to-all sends them back. This is the biggest difference between EP and TP/DP — EP uses all-to-all, not all-reduce.

DeepSeek-V3 uses 256 experts with 8 chosen per token, and EP=64 distributes the experts across 64 cards; Mixtral 8x7B uses 8 experts with EP=8. EP can also be combined with TP: each card does TP internally. For example, EP=4 x TP=2 uses 8 cards to manage 4 experts, with each expert's weights sharded across 2 cards. The hand calculation below works through the all-to-all communication volume of EP.


In [ ]:
# === EP mock: 4 experts, 4 cards, 1 expert per card ===
num_experts = 4
world_size  = 4
seq_len     = 8
d_model     = 4

np.random.seed(1)
# Each token is routed to some expert
tokens = np.random.randn(seq_len, d_model)
router_logits = tokens @ np.random.randn(d_model, num_experts)
assignments = router_logits.argmax(axis=1)   # expert id chosen for each token

print(f"token -> expert routing: {assignments.tolist()}")
print()

# Simulate all-to-all dispatch: each card (expert) gathers the tokens sent to it
expert_inputs = [[] for _ in range(num_experts)]
for tok_idx, expert_id in enumerate(assignments):
    expert_inputs[expert_id].append(tokens[tok_idx])

for eid in range(num_experts):
    n_tokens = len(expert_inputs[eid])
    print(f"expert {eid} (card {eid}) received {n_tokens} tokens")
print()
print("Key observation: each token is sent to exactly one card; the all-to-all volume = seq_len x d_model.")
print("After computation, a reverse all-to-all sends the results back to each token's original location.")


In [ ]:
# === EP vs TP vs DP sharding comparison on MoE ===
# Assume 8 experts x 110M params per expert MLP, 8 cards
expert_params = 110e6
num_experts   = 8
total_params  = expert_params * num_experts
N             = 8

ep_per_card = expert_params     # EP: 1 expert per card
tp_per_card = total_params / N  # TP: every expert sharded into 8, 1/8 per card per expert
dp_per_card = total_params      # DP: full replica per card

print(f"{'Scheme':<10}{'Per-card expert weights (M)':>28}{'Post-routing comm':>22}")
print("-" * 62)
print(f"{'EP':<10}{ep_per_card/1e6:>26.1f}{'all-to-all':>22}")
print(f"{'TP':<10}{tp_per_card/1e6:>26.1f}{'all-reduce':>22}")
print(f"{'DP':<10}{dp_per_card/1e6:>26.1f}{'all-reduce (grad)':>22}")
print()
print("EP is the most memory-efficient per card, but its all-to-all communication pattern differs from all-reduce and demands more of the network topology.")
print("Large models usually mix EP + DP + TP; Section 6 expands on this.")


## 6. Composing 5D Parallelism

Stitching the five parallelism dimensions together yields the "parallelism recipe" used in industrial training. A 2048-card cluster is typically organized as a 3D or 4D mesh: the innermost 8 cards do TP (within the NVLink domain), a group outward does EP or PP (across nodes), and another level outward does DP. Three publicly reported configurations follow:

- **Llama-3 70B**: TP=8 within an 8-card node, PP=8 across nodes, DP for the remaining dimension, totaling around 1024 GPUs. A dense model with no EP.
- **Mixtral 8x7B**: 8 experts with EP=8 (one expert per card), TP=1 within a node, DP across nodes, trained on roughly 128-256 cards.
- **DeepSeek-V3 671B (MoE, 37B activated)**: EP=64 (256 experts across 64 cards, 4 per card), TP=1 (no intra-node TP because MLA already keeps per-layer weights small enough), PP=16 with DualPipe, DP filling the remaining GPUs, for a total of 2048 H800 GPUs.

There are three composition principles: TP is used only within a node (all-reduce is latency-sensitive); EP pairs with MoE and matches the expert count to the card count; PP and DP fill out the remaining GPUs and ensure enough micro-batches to hide the bubble. The larger the model and the more experts, the larger the share taken by EP.


In [ ]:
# === Comparison of typical parallelism recipes ===
configs = [
    ("Llama-3 70B",   "dense",  {"TP": 8, "PP": 8,  "DP": 16, "EP": 1,  "SP": 1}),
    ("Mixtral 8x7B",  "MoE",    {"TP": 1, "PP": 1,  "DP": 16, "EP": 8,  "SP": 1}),
    ("DeepSeek-V3",   "MoE",    {"TP": 1, "PP": 16, "DP": 2,  "EP": 64, "SP": 1}),
]

print(f"{'Model':<18}{'Type':<8}{'TP':>4}{'PP':>5}{'DP':>5}{'EP':>5}{'SP':>5}{'Total GPU':>12}")
print("-" * 64)
for name, kind, c in configs:
    total = c["TP"] * c["PP"] * c["DP"] * c["EP"] * c["SP"]
    print(f"{name:<18}{kind:<8}{c['TP']:>4}{c['PP']:>5}{c['DP']:>5}{c['EP']:>5}{c['SP']:>5}{total:>12}")
print()
print("Note: DeepSeek-V3's EP=64 and DP=2 are orthogonal dimensions. "
      "Inside an EP group, 64 cards split the 256 experts; across DP groups the whole group is replicated.")
print("Total GPU = TP x PP x DP x EP x SP, the product of the five dimensions.")


In [ ]:
# === Typical recipes for different model scales ===
recipes = [
    ("7B dense",    {"TP": 1, "PP": 1, "DP": 8,  "EP": 1}),
    ("70B dense",  {"TP": 8, "PP": 4,  "DP": 4,  "EP": 1}),
    ("400B MoE",   {"TP": 4, "PP": 8,  "DP": 4,  "EP": 16}),
    ("1T+ MoE",    {"TP": 4, "PP": 16, "DP": 8,  "EP": 32}),
]

print(f"{'Scale':<16}{'TP':>4}{'PP':>5}{'DP':>5}{'EP':>5}{'Total GPU':>12}")
print("-" * 52)
for name, c in recipes:
    total = c["TP"] * c["PP"] * c["DP"] * c["EP"]
    print(f"{name:<16}{c['TP']:>4}{c['PP']:>5}{c['DP']:>5}{c['EP']:>5}{total:>12}")
print()
print("Pattern: the larger the model, the higher the share of PP and EP; TP is usually locked at 4 or 8 within a node.")
print("DP is used to scale the batch, but cannot be increased without limit due to the critical batch size.")


## Summary

Confirm that you understand the following:

- [ ] DDP splits the batch, with an all-reduce at the end of the backward; the three ZeRO stages have identical communication volume, just trading reclaimed bandwidth for memory
- [ ] Tensor Parallelism chains column parallel with row parallel, so an entire MLP/Attention layer needs only one forward all-reduce
- [ ] Splitting attention by head is naturally communication-free, because different heads are independent
- [ ] The bubble fraction of Pipeline Parallelism is $(P-1)/M$; 1F1B and DualPipe push it lower still
- [ ] Sequence Parallelism splits the sequence dimension, saving activation memory alongside TP and serving long-context training on its own
- [ ] Expert Parallelism splits the expert dimension; its core communication is all-to-all, different from the all-reduce used by TP/DP
- [ ] 5D parallelism = DP x TP x PP x SP x EP; the total GPU count equals the product of the dimensions
- [ ] TP locked within a node, with PP/EP/DP filling the cross-node dimension, is the general principle of industrial recipes


## Exercises

> You may ask AI to help explain the ideas, but it is not advisable to ask AI to "solve this exercise for you" outright.

**Exercise 1: Hand-compute per-card memory for a 7B model under TP=2 + DP=2**

A 7B dense model trained with AdamW + BF16 on 4 cards. Configure TP=2, DP=2, no ZeRO. Each card holds the full optimizer state (a DP replica) but the weights are sharded by TP. What is the fixed per-card memory in GB?

Hint: TP=2 shards each layer's weights across 2 cards, so the parameter portion per card is $2P/2$; the same goes for gradients. Without ZeRO the optimizer-state portion is still $12P$. Total: $2P/2 + 2P/2 + 12P = 14P$ bytes.


In [ ]:
# Exercise 1: per-card memory of a 7B model under TP=2 + DP=2
P = 7e9

# TODO: compute the per-card memory (bytes)
# TP=2 shards weights and gradients; DP=2 does not shard optimizer state
tp_per_card_bytes = None

assert tp_per_card_bytes is not None, "Compute the per-card memory first"
expected = 2 * P / 2 + 2 * P / 2 + 12 * P   # = 14P
assert abs(tp_per_card_bytes - expected) < 1e9, f"Should be {expected / 1e9:.0f} GB"
print(f"Exercise 1 passed:")
print(f"   Per-card memory of a 7B model under TP=2 + DP=2 = {tp_per_card_bytes / 1e9:.0f} GB")
print(f"   Versus 112 GB for pure DDP, this saves {(112 - tp_per_card_bytes/1e9):.0f} GB (weights halved).")


**Exercise 2: Compute the bubble fraction for PP=8, M=32**

Hint: bubble = (P-1)/M. Substitute P=8, M=32.


In [ ]:
# Exercise 2: PP bubble fraction
P, M = 8, 32

# TODO: compute the bubble fraction (a decimal between 0 and 1)
bubble_ratio = None

assert bubble_ratio is not None, "Compute the bubble fraction first"
expected = (P - 1) / M
assert abs(bubble_ratio - expected) < 1e-6, f"Should be {expected:.4f}"
print(f"Exercise 2 passed:")
print(f"   With PP={P}, M={M}, the bubble fraction = {bubble_ratio*100:.1f}%")
print(f"   Switching to DualPipe (bidirectional) doubles the effective M, dropping the bubble to {bubble_ratio*50:.1f}%.")


**Exercise 3: Fill in the total GPU count for a DeepSeek-V3-style 5D parallel config**

DeepSeek-V3 is configured with TP=1, PP=16, DP=2, EP=64, SP=1. Compute the total number of GPUs.

Hint: total GPU for 5D parallelism = TP x PP x DP x EP x SP.


In [ ]:
# Exercise 3: total GPU count for DeepSeek-V3
config = {"TP": 1, "PP": 16, "DP": 2, "EP": 64, "SP": 1}

# TODO: compute the total number of GPUs
total_gpus = None

assert total_gpus is not None, "Compute the total GPU count first"
expected = config["TP"] * config["PP"] * config["DP"] * config["EP"] * config["SP"]
assert total_gpus == expected, f"Should be {expected}"
print(f"Exercise 3 passed:")
print(f"   DeepSeek-V3 config {config}")
print(f"   Total GPU = {total_gpus} (publicly reported as 2048 H800s)")
print(f"   EP=64 accounts for most of the cards, since 256 experts across 64 cards is 4 experts per card.")


## References

- Shoeybi et al., [Megatron-LM: Training Multi-Billion Parameter Language Models Using Model Parallelism](https://arxiv.org/abs/1909.08053), 2019
- Huang et al., [GPipe: Efficient Training of Giant Neural Networks using Pipeline Parallelism](https://arxiv.org/abs/1811.06965), 2018
- Narayanan et al., [Memory-Efficient Pipeline-Parallel DNN Training (1F1B)](https://arxiv.org/abs/2004.13378), 2020
- DeepSeek-AI, [DeepSeek-V3 Technical Report (DualPipe)](https://arxiv.org/abs/2412.19437), 2024
- Korthikanti et al., [Reducing Activation Recomputation in Large Transformer Models (Sequence Parallelism)](https://arxiv.org/abs/2205.05198), 2022
- Fedus et al., [Switch Transformers (Expert Parallelism)](https://arxiv.org/abs/2101.03961), 2021
- Liu et al., [Ring Attention with Blockwise Transformers for Near-Infinite Context](https://arxiv.org/abs/2310.01889), 2023